<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/right_predictions/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import pickle
from google.colab import drive

drive.mount('/content/drive')

# tüm veri
with open("/content/drive/MyDrive/datasets/merged_europarl_informal.pkl", "rb") as f:
    merged_df = pd.read_pickle(f)

# yanlış tahmin edilen veriler
with open("/content/drive/MyDrive/datasets/balanced_wrong_predictions_v2.pkl", "rb") as f:
    wrong_df = pd.read_pickle(f)

# text + gender kombinasyonunu kullanarak wrong'ları çıkar
wrong_pairs = set(zip(wrong_df["text"], wrong_df["gender"]))
right_df = merged_df[~merged_df[["text", "gender"]].apply(tuple, axis=1).isin(wrong_pairs)].copy()

# ⛔️ Aynı text farklı gender'larla geçmişse — bu text'leri tamamen çıkar
duplicated_texts = (
    right_df.groupby("text")["gender"]
    .nunique()
    .reset_index()
    .query("gender > 1")["text"]
    .tolist()
)

right_df = right_df[~right_df["text"].isin(duplicated_texts)].copy()

# ✅ Aynı text'ten sadece 1 tanesini tut (her text sadece bir satır olacak)
right_df = right_df.drop_duplicates(subset="text")

# index'i sıfırla
right_df = right_df.reset_index(drop=True)

# kaydet
right_path = "/content/drive/MyDrive/datasets/right_precision.pkl"
with open(right_path, "wb") as f:
    pickle.dump(right_df, f)

print(f"[INFO] Doğru tahmin edilen, çakışmasız ve unique text içeren veriler kaydedildi: {right_path}")
print(right_df["gender"].value_counts())

Mounted at /content/drive
[INFO] Doğru tahmin edilen, çakışmasız ve unique text içeren veriler kaydedildi: /content/drive/MyDrive/datasets/right_precision.pkl
gender
male      418796
female    405476
Name: count, dtype: int64


In [2]:
print(wrong_df["gender"].value_counts())

gender
female    163210
male      163210
Name: count, dtype: int64


In [3]:
dup = merged_df.groupby("text")["gender"].nunique()
print(dup[dup > 1])

text
" Anyway, if there's a potential for a good to great relationship, then don't squander the chance because you're too much of a @#%$ to throw your balls in the ring. Regrets are a bitch, especially when you don't get play from them ." -evasivecolin  This is something entirely unrelated that an old friend/acquaintance of mine posted.  It made me smile - and seemed oddly appropriate in my life, so here it is.  Thought it should get more publication, and public notice . . .so now a good five of us will see it, no? ^_^  I'm tired.  I started work today - it went, as things often do, better than feared and more mundane than hoped for.  Would you like to donate a dollar to the Save the Children fund?  "Two-dollah"  What?!  Tomorrow? "What children are we saving?"  Um, ma'am, I really don't know. "Did you know only 15% of that goes to the children?"  No sir, really?  Do you want to donate a dollar to the Save the Children overhead?    2
" HiEmDosHuevos:  Glah.  HiEmDosHuevos:  Things driv

In [4]:
# Aynı text'ten kaç tane olduğunu say
text_counts = right_df["text"].value_counts()

# Sadece 2 kez geçenleri filtrele
twice_appeared = text_counts[text_counts == 2]

# Kaç tane böyle text var?
print(f"2 kere geçen text sayısı: {len(twice_appeared)}")

# Eğer örnek görmek istersen:
print(twice_appeared.head(10))  # ilk 10 tanesini yazdır

2 kere geçen text sayısı: 0
Series([], Name: count, dtype: int64)


In [7]:
# Her text'in kaç farklı gender'a sahip olduğunu kontrol et
dup_gender = (
    right_df.groupby("text")["gender"]
    .nunique()
)

# Sadece birden fazla gender'la eşleşen text'leri filtrele
conflicting_texts = dup_gender[dup_gender > 1]

# Sonuç
print(f"Farklı gender'larda geçen text sayısı: {len(conflicting_texts)}")

Farklı gender'larda geçen text sayısı: 0
